### Notebook de processing

Compila todos los `*_preprocessed_long.csv` generados por `01_preprocessing.ipynb` (una fila por ROI/frame/TTL, con `phase`, `trend` y `ROI_status` ya fijados) en un unico DataFrame para los pasos siguientes de analisis.

No lee ningun Excel: recorre directamente las carpetas de muestra dentro de `base_dir`.

In [1]:
%matplotlib qt
from pathlib import Path
import sys
import matplotlib.pyplot as plt

sys.path.append(str(Path("../scripts").resolve()))
import analisis_ttl as ttl

base_dir = Path("/Users/gfernandezv/Documents/envs/Images_TTL_temp/data/Proc_data")

In [2]:
imports = ttl.load_all_preprocessed_long(base_dir)

load_status = imports["load_status"]
preprocessed_all = imports["preprocessed_all"]

print(f"preprocessed_all: {preprocessed_all.shape}")
if preprocessed_all.empty:
    print("No se encontraron archivos *_preprocessed_long.csv. Ejecuta primero 01_preprocessing.ipynb para cada muestra.")

# Cada carpeta debe tener un unico *_preprocessed_long.csv. Mas de uno
# generalmente significa que quedo un CSV viejo sin borrar de una corrida
# anterior de 01_preprocessing.ipynb (ver notas de esa celda de exportacion),
# y se estarian duplicando ROIs de esa muestra en preprocessed_all.
duplicated_preprocessed = load_status[load_status["n_files"] > 1]
if not duplicated_preprocessed.empty:
    print("ADVERTENCIA: carpetas con mas de un *_preprocessed_long.csv (revisar y dejar solo el correcto):")
    print(duplicated_preprocessed[["folder", "preprocessed_files"]].to_string(index=False))

load_status

preprocessed_all: (236630, 43)


,folder,preprocessed_files,n_files,status
0,mut27_image10,[sample_10_m27_cooling_preprocessed_long.csv],1,ok
1,mut27_image11,[sample_11_m27_cooling_preprocessed_long.csv],1,ok
2,mut36_image8,[sample_08_m36_cooling_preprocessed_long.csv],1,ok
3,mut45_image13,[sample_13_m45_cooling_preprocessed_long.csv],1,ok
4,mut57_image2,[sample_02_m57_cooling_preprocessed_long.csv],1,ok
5,mut57_image3,[sample_03_m57_cooling_preprocessed_long.csv],1,ok
6,mut57_image4,[sample_04_m57_cooling_preprocessed_long.csv],1,ok
7,mut65_image19,[sample_19_m65_cooling_preprocessed_long.csv],1,ok


### Intensidad normalizada de ROI seleccionadas, por muestra

`preprocessed_all` mezcla todas las muestras, y el nombre de ROI se repite entre carpetas (ej. `ROI1` existe en cada una). Por eso el filtro para graficar necesita `folder` (`source_folder`) como variable: identifica de qué muestra son las ROI que se están graficando.

In [3]:
# Carpetas disponibles (source_folder) en el compilado.
sorted(preprocessed_all["source_folder"].dropna().unique())

['mut27_image10',
 'mut27_image11',
 'mut36_image8',
 'mut45_image13',
 'mut57_image2',
 'mut57_image3',
 'mut57_image4',
 'mut65_image19']

In [4]:
# Muestra a trabajar (una de source_folder listadas arriba). folder e input_dir
# quedan atados: todo lo de abajo (selected_rois, graficos, reprocesar) usa
# esta misma carpeta, para no desincronizar cual muestra se esta mirando.
#
# folder debe ser el nombre EXACTO de una carpeta, no un patron (ej. "mut27_*").
# input_dir = base_dir / folder necesita una carpeta real para leer el ABF/CSV
# crudo; para reprocesar varias muestras con un patron, usa la celda de
# "Reprocesar varias muestras a la vez (batch)" mas abajo en su lugar.
folder = "mut45_image13"

input_dir = base_dir / folder
if any(ch in folder for ch in "*?["):
    raise ValueError(
        f"folder={folder!r} parece un patron, no una carpeta exacta. "
        "selected_rois_from_long/graph_selected_rois_by_folder si aceptan patrones, "
        "pero input_dir (para process_sample) necesita una carpeta real: "
        "usa la celda de reprocesamiento batch (mas abajo) para varias muestras."
    )
if not input_dir.exists():
    raise ValueError(f"No existe la carpeta: {input_dir}")

In [5]:
# ROIs con ROI_status=1 para "folder", leidas directamente del _long compilado
# (sin recortes de imagen ni listas escritas a mano). Sirve, por ejemplo, como
# selected_rois para ttl.process_sample() si se quiere reprocesar esa muestra.
selected_rois = ttl.selected_rois_from_long(preprocessed_all, folder=folder)
selected_rois

mut45_image13: 13 ROIs con ROI_status=1


['ROI2',
 'ROI8',
 'ROI10',
 'ROI17',
 'ROI28',
 'ROI32',
 'ROI35',
 'ROI41',
 'ROI73',
 'ROI82',
 'ROI84',
 'ROI85',
 'ROI98']

### Reprocesar varias muestras a la vez (batch)

`process_sample()` solo lee un ABF/CSV crudo por llamada, así que un `folder` con patrón (ej. `"mut27_*"`) no le sirve directamente. `ttl.process_selected_folders()` resuelve esto: recorre una lista de carpetas con un `for`, deriva `selected_rois` de cada una por separado (sin mezclar ROI del mismo nombre entre muestras) y junta los `df_phase` en una sola tabla con `source_folder`.

In [6]:
import fnmatch

# Patron de carpetas a reprocesar juntas, ej. "mut27_*" para todas las m27.
folder_pattern = "mut57_*"
batch_folders = fnmatch.filter(sorted(preprocessed_all["source_folder"].unique()), folder_pattern)
print("Carpetas a reprocesar:", batch_folders)

batch_result = ttl.process_selected_folders(
    base_dir,
    batch_folders,
    preprocessed_all,
    start_ttl=0,
    threshold=2.0,
    target_temp=22,
    norm_type=1,
    phase_filter="cooling",
    temp_range=(20, 40),
)
df_phase_batch = batch_result["df_phase_all"]

plotted_batch = ttl.graph_selected_rois_by_folder(
    df_phase_batch,
    folder=folder_pattern,
    value_col="NormSignal",
    x_col="temp_mean",
    show_mean_se=True,
)

Carpetas a reprocesar: ['mut57_image2', 'mut57_image3', 'mut57_image4']


mut57_image2: 8 ROIs con ROI_status=1
Archivos encontrados en: /Users/gfernandezv/Documents/envs/Images_TTL_temp/data/Proc_data/mut57_image2
  ROI CSV:
    - ROI_Results.csv
  ABF:
    - 26423001.abf


Procesamiento completo:
  eventos TTL: 488
  frames CSV total: 200
  frames con TTL asociado: 200
  frames sin TTL asociado: 0
  filtro de fase previo a normalizar: cooling
  filtro de rango de temperatura previo a normalizar: (20, 40)
  frames usados para normalizar: 137
  TTL_index asignados a frames: 0 a 199
  TTL_index detectados: 0 a 487
  filas df_norm: 1096
  ROIs: 8
  fases:
phase
cooling    137
Name: count, dtype: int64
  tiempo total (s): 987.26
  tiempo entre frames (mediana, s): 2.0000
  tiempo entre frames (media, s): 1.9999


mut57_image3: 10 ROIs con ROI_status=1
Archivos encontrados en: /Users/gfernandezv/Documents/envs/Images_TTL_temp/data/Proc_data/mut57_image3
  ROI CSV:
    - ROI_Results_2.csv
  ABF:
    - 26423003.abf


Procesamiento completo:
  eventos TTL: 663
  frames CSV total: 662
  frames con TTL asociado: 662
  frames sin TTL asociado: 0
  filtro de fase previo a normalizar: cooling
  filtro de rango de temperatura previo a normalizar: (20, 40)
  frames usados para normalizar: 341
  TTL_index asignados a frames: 0 a 661
  TTL_index detectados: 0 a 662
  filas df_norm: 3410
  ROIs: 10
  fases:
phase
cooling    341
Name: count, dtype: int64
  tiempo total (s): 1335.94
  tiempo entre frames (mediana, s): 2.0000
  tiempo entre frames (media, s): 1.9999


mut57_image4: 153 ROIs con ROI_status=1
Archivos encontrados en: /Users/gfernandezv/Documents/envs/Images_TTL_temp/data/Proc_data/mut57_image4
  ROI CSV:
    - ROI_Results.csv
  ABF:
    - 26423004.abf


ROI1: sin datos cerca de 22°C
ROI2: sin datos cerca de 22°C
ROI3: sin datos cerca de 22°C
ROI4: sin datos cerca de 22°C
ROI5: sin datos cerca de 22°C
ROI6: sin datos cerca de 22°C
ROI7: sin datos cerca de 22°C
ROI8: sin datos cerca de 22°C
ROI9: sin datos cerca de 22°C
ROI10: sin datos cerca de 22°C
ROI11: sin datos cerca de 22°C
ROI12: sin datos cerca de 22°C
ROI13: sin datos cerca de 22°C
ROI14: sin datos cerca de 22°C
ROI15: sin datos cerca de 22°C
ROI16: sin datos cerca de 22°C
ROI17: sin datos cerca de 22°C
ROI18: sin datos cerca de 22°C
ROI19: sin datos cerca de 22°C
ROI20: sin datos cerca de 22°C
ROI21: sin datos cerca de 22°C
ROI22: sin datos cerca de 22°C
ROI23: sin datos cerca de 22°C
ROI24: sin datos cerca de 22°C
ROI25: sin datos cerca de 22°C
ROI26: sin datos cerca de 22°C
ROI27: sin datos cerca de 22°C
ROI28: sin datos cerca de 22°C
ROI29: sin datos cerca de 22°C
ROI30: sin datos cerca de 22°C
ROI31: sin datos cerca de 22°C
ROI32: sin datos cerca de 22°C
ROI33: sin datos 

ROI88: sin datos cerca de 22°C
ROI89: sin datos cerca de 22°C
ROI90: sin datos cerca de 22°C
ROI91: sin datos cerca de 22°C
ROI92: sin datos cerca de 22°C
ROI93: sin datos cerca de 22°C
ROI94: sin datos cerca de 22°C
ROI95: sin datos cerca de 22°C
ROI96: sin datos cerca de 22°C
ROI97: sin datos cerca de 22°C
ROI98: sin datos cerca de 22°C
ROI99: sin datos cerca de 22°C
ROI100: sin datos cerca de 22°C
ROI101: sin datos cerca de 22°C
ROI102: sin datos cerca de 22°C
ROI103: sin datos cerca de 22°C
ROI104: sin datos cerca de 22°C
ROI105: sin datos cerca de 22°C
ROI106: sin datos cerca de 22°C
ROI107: sin datos cerca de 22°C
ROI108: sin datos cerca de 22°C
ROI109: sin datos cerca de 22°C
ROI110: sin datos cerca de 22°C
ROI111: sin datos cerca de 22°C
ROI112: sin datos cerca de 22°C
ROI113: sin datos cerca de 22°C
ROI114: sin datos cerca de 22°C
ROI115: sin datos cerca de 22°C
ROI116: sin datos cerca de 22°C
ROI117: sin datos cerca de 22°C
ROI118: sin datos cerca de 22°C
ROI119: sin datos ce

mut57_*: 153 ROIs graficadas, 30975 filas


/Users/gfernandezv/Documents/envs/Images_TTL_temp/scripts/analisis_ttl.py:505: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()
